In [6]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_huggingface import HuggingFaceEmbeddings
import os

In [7]:
load_dotenv()

True

In [8]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

In [9]:
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [10]:
index_name = "prod-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [11]:
file_path = "D:\ProdRAG\prodRAG\example.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [13]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 687.60it/s]


In [14]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [15]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
query = "What is the module4?"
results = retriever.invoke(query)

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
def chunks_docs(docs):
    return [doc.page_content for doc in docs]

In [18]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [19]:
rag_chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
        "chunks_context": retriever | RunnableLambda(chunks_docs),
    }
)

In [20]:
result=rag_chain.invoke("What is 4th module?")

In [21]:
print(result)

{'question': 'What is 4th module?', 'context': 'blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.\n\nnetworks, and edge devices using blockchain. \nModule 7: Cybersecurity and Blockchain — Role of blockchain in identity management, \nthreat detection, and data security. \nModule 8: Case Studies and Project Work — Applications of blockchain in supply chain, \nhealthcare, and finance. \n5. Practical Components \n- Setting up a private Ethereum blockchain network. \n- Creating and deploying smart contracts using Solidity. \n- Developing a basic decentralized